# Recompute match_source: Query NASA Disposition

**Logic**: For each target, query NASA to get its actual **disposition** — not just whether it exists in a table.

| Disposition | Meaning |
|---|---|
| `CONFIRMED` | NASA has confirmed this as a real planet |
| `CANDIDATE` | Passed vetting but not yet confirmed |
| `FALSE POSITIVE` | NASA has ruled it out as non-planet |
| `none` | Not found in any NASA table |

**Method**: TAP query to NASA Exoplanet Archive `cumulative` table (Kepler/K2) + `toi` table (TESS), matching by target name and TIC ID.

**Input**: `merge_results.CSV`

**Output**: Same CSV with `match_source` overwritten to the NASA disposition.

In [1]:
%pip install -q numpy pandas astropy astroquery requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings, requests
from io import StringIO
import numpy as np
import pandas as pd
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

warnings.filterwarnings('ignore')

df = pd.read_csv('01.CSV')
print(f'Loaded: {len(df)} rows')
df.head(3)

Loaded: 50 rows


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive,depth_odd_ppm,depth_even_ppm,...,odd_even_pass,TIC_ID,Stellar_Radius_Rsun,Teff_K,Spectral_Type,Rp_Rjup,Rp_Rearth,radius_reason,planet_radius_plausible,match_source
0,L 98-59,2,3.68989,0.00055,685.0,1.3,13.43,False,1227.087681,1367.776127,...,True,307210830,0.314160,3429.0,M,0.080013,0.896853,plausible,True,confirmed
1,TOI-700,1,11.34365,0.28837,1283.0,0.5,5.10,True,3275.255154,1012.759695,...,False,150428135,0.419457,3494.0,M,0.146205,1.638799,plausible,True,confirmed
2,TOI-270,3,11.32061,0.12267,1531.5,1.3,10.14,False,2976.994045,2304.645538,...,True,259377017,0.374358,3532.0,M,0.142563,1.597977,plausible,True,confirmed


In [3]:
TAP_URL = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync'
_CACHE = {}

def _okt(x):
    try: return x is not None and not pd.isna(x) and int(x) > 0
    except: return False

def query_nasa_disposition(target, tic_id=None):
    '''Return one of: CONFIRMED, CANDIDATE, FALSE POSITIVE, none'''
    t = str(target)
    key = t
    if key in _CACHE:
        return _CACHE[key]

    # ---- (1) Query cumulative table (Kepler/K2) by target name ----
    try:
        q = f"select koi_disposition from cumulative where kepoi_name='{t}'"
        resp = requests.get(TAP_URL, params={'query': q, 'format': 'csv'}, timeout=30)
        if resp.status_code == 200:
            txt = resp.text.strip()
            if txt and 'koi_disposition' in txt:
                val = txt.split('\n')[1].strip() if '\n' in txt else ''
                if val:
                    _CACHE[key] = val
                    return val
    except Exception:
        pass

    # ---- (2) Query ps (confirmed planets) by hostname ----
    try:
        q = f"select pl_name,pl_orbper from ps where hostname='{t}'"
        resp = requests.get(TAP_URL, params={'query': q, 'format': 'csv'}, timeout=30)
        if resp.status_code == 200 and 'pl_name' in resp.text and len(resp.text.strip().split('\n')) > 1:
            _CACHE[key] = 'CONFIRMED'
            return 'CONFIRMED'
    except Exception:
        pass

    # ---- (3) Query TOI table by TIC ID ----
    if _okt(tic_id):
        try:
            q = f"select tfopwg_disp from toi where tid={int(tic_id)}"
            resp = requests.get(TAP_URL, params={'query': q, 'format': 'csv'}, timeout=30)
            if resp.status_code == 200:
                txt = resp.text.strip()
                if txt and 'tfopwg_disp' in txt:
                    lines = txt.split('\n')
                    if len(lines) > 1:
                        disp = lines[1].strip()
                        # Map TOI dispositions to standard categories
                        if disp in ('CP',):
                            _CACHE[key] = 'CONFIRMED'
                        elif disp in ('KP', 'PC', 'APC'):
                            _CACHE[key] = 'CANDIDATE'
                        elif disp in ('FP', 'EB', 'NTP', 'V', 'FA'):
                            _CACHE[key] = 'FALSE POSITIVE'
                        else:
                            _CACHE[key] = 'CANDIDATE'
                        return _CACHE[key]
        except Exception:
            pass

    # ---- (4) Try NasaExoplanetArchive.query_object as fallback ----
    try:
        result = NasaExoplanetArchive.query_object(t)
        if result is not None and len(result) > 0:
            _CACHE[key] = 'CONFIRMED'
            return 'CONFIRMED'
    except Exception:
        pass

    _CACHE[key] = 'none'
    return 'none'

print('Function ready.')

Function ready.


In [4]:
# Diagnostic: first 3 targets
print('=== Diagnostic ===')
for i in range(min(3, len(df))):
    row = df.iloc[i]
    tgt, tic = row['Target'], row.get('TIC_ID', None)
    disp = query_nasa_disposition(tgt, tic)
    print(f"  {tgt:<25} TIC={tic} -> {disp}")

=== Diagnostic ===
  L 98-59                   TIC=307210830 -> CONFIRMED
  TOI-700                   TIC=150428135 -> CONFIRMED
  TOI-270                   TIC=259377017 -> CONFIRMED


In [5]:
sources = []
for i, row in df.iterrows():
    tgt = row['Target']
    tic = row.get('TIC_ID', None)
    disp = query_nasa_disposition(tgt, tic)
    sources.append(disp)
    print(f"[{i+1:>3}/{len(df)}] {str(tgt)[:25]:<25} -> {disp}")

df['match_source'] = sources
df.to_csv('merge_results.CSV', index=False, encoding='utf-8-sig')

from collections import Counter
print(f"\nDone. Distribution: {dict(Counter(sources))}")

[  1/50] L 98-59                   -> CONFIRMED
[  2/50] TOI-700                   -> CONFIRMED
[  3/50] TOI-270                   -> CONFIRMED
[  4/50] GJ 357                    -> CONFIRMED
[  5/50] HD 21749                  -> CANDIDATE
[  6/50] TOI-125                   -> CONFIRMED
[  7/50] TOI-396                   -> CANDIDATE
[  8/50] TOI-451                   -> CONFIRMED
[  9/50] TOI-561                   -> CONFIRMED
[ 10/50] TOI-620                   -> CONFIRMED
[ 11/50] LTT 1445                  -> none
[ 12/50] TOI-776                   -> CONFIRMED
[ 13/50] TOI-824                   -> CONFIRMED
[ 14/50] TOI-1235                  -> CONFIRMED
[ 15/50] TOI-1634                  -> CONFIRMED
[ 16/50] TOI-1685                  -> CONFIRMED
[ 17/50] TOI-2136                  -> CONFIRMED
[ 18/50] TOI-2285                  -> CONFIRMED
[ 19/50] GJ 3473                   -> CONFIRMED
[ 20/50] TOI-178                   -> CONFIRMED
[ 21/50] TOI-431                   -> CONFIRM